# ISIC external check: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import io
import glob
import time
import shutil
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# GPU guard

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. The ISIC arm trains ResNet-18 across 45 runs; "
        "a CPU kernel is infeasible. Switch the Kaggle accelerator to GPU (P100) and rerun. "
        "Notebook 00 can run on CPU, notebook 05 cannot."
    )
print("CUDA available:", torch.cuda.get_device_name(0))

CUDA available: Tesla T4


# Stage the frozen config and refresh (ISIC inherits the frozen PCam training config)

In [4]:
pulled_ok = False
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        pulled_ok = True
        print("pulled results store (latest) from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("PULL FAILED:", exc)

if not pulled_ok:
    raise RuntimeError(
        "pull failed. This session would otherwise push a fresh store over the "
        "existing one and clobber prior protocols. Fix the pull and rerun. Do not "
        "let the sweep or push proceed without the prior ledger in place."
    )

if not os.path.exists(core.PATHS["ledger"]):
    raise RuntimeError(
        f"pull reported success but no ledger at {core.PATHS['ledger']}. The store "
        "is empty or malformed. Stop and check the dataset before running the sweep."
    )

prior = pd.read_csv(core.PATHS["ledger"])
protocols = sorted(set("|".join(prior["protocol"].astype(str)).split("|")))
print(f"prior ledger rows: {len(prior)}")
print("protocols present:", protocols)

required = {"protocolA", "protocolB", "protocolC", "protocolD"}
missing = required - set(protocols)
if missing:
    raise RuntimeError(
        f"pulled ledger is missing {missing}. ISIC should run after all four PCam "
        "protocols are in the store. Confirm 01-04 finished and pushed, then rerun."
    )
print("prior state confirmed: A, B, C, D present")

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "frozen_config.md not found in the working dir or any mounted dataset. "
            "Run 00_setup.ipynb and mount the results store before running this notebook."
        )
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN (PCam, not used for ISIC):", core.NORM_MEAN)

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store (latest) from hrushisanap/datawise-results
prior ledger rows: 225
protocols present: ['protocolA', 'protocolB', 'protocolC', 'protocolD']
prior state confirmed: A, B, C, D present
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN (PCam, not used for ISIC): [0.700756, 0.538358, 0.691621]


# Locate the ISIC files by recursive search

In [5]:
gt_hits = glob.glob("/kaggle/input/**/ISIC_2020_Training_GroundTruth_v2.csv", recursive=True)
dup_hits = glob.glob("/kaggle/input/**/ISIC_2020_Training_Duplicates.csv", recursive=True)
if not gt_hits:
    raise FileNotFoundError("ISIC ground-truth CSV not found under /kaggle/input")
if not dup_hits:
    raise FileNotFoundError("ISIC duplicates CSV not found under /kaggle/input")
gt_path = gt_hits[0]
dup_path = dup_hits[0]

img_hits = glob.glob("/kaggle/input/**/ISIC_2020/train/*.jpg", recursive=True)
if not img_hits:
    img_hits = glob.glob("/kaggle/input/**/train/ISIC_*.jpg", recursive=True)
if not img_hits:
    raise FileNotFoundError("ISIC image directory not found under /kaggle/input")
img_dir = os.path.dirname(img_hits[0])

print("ground truth:", gt_path)
print("duplicates:  ", dup_path)
print("image dir:   ", img_dir, f"({len(img_hits)} jpg files)")

ground truth: /kaggle/input/datasets/bandookdaddy/datawise-isic2020/ISIC_2020_Training_GroundTruth_v2.csv
duplicates:   /kaggle/input/datasets/bandookdaddy/datawise-isic2020/ISIC_2020_Training_Duplicates.csv
image dir:    /kaggle/input/datasets/bandookdaddy/datawise-isic2020/ISIC_2020/train (33126 jpg files)


# Build the deduped pool and the frozen holdout

In [6]:
ISIC_HOLDOUT_SEED = 20200

gt = pd.read_csv(gt_path)
dup = pd.read_csv(dup_path)

drop_names = set(dup["image_name_2"].astype(str))
deduped = gt[~gt["image_name"].astype(str).isin(drop_names)].reset_index(drop=True)

n_dedup = len(deduped)
n_mal = int((deduped["target"] == 1).sum())
n_ben = int((deduped["target"] == 0).sum())
print(f"deduped pool: {n_dedup} (malignant {n_mal} / benign {n_ben})")

assert n_dedup == 32701, f"deduped pool {n_dedup} != 32,701 (mount changed)"
assert n_mal == 581, f"malignant {n_mal} != 581"
assert n_ben == 32120, f"benign {n_ben} != 32,120"
assert n_mal >= 184 + 384, "not enough malignant for holdout + fixed minority"
print(f"malignant margin after holdout + fixed minority: {n_mal - 184 - 384}")

targets = deduped["target"].to_numpy().astype(int)
names = deduped["image_name"].astype(str).to_numpy()

order = np.random.default_rng(ISIC_HOLDOUT_SEED).permutation(n_dedup)
hold_pos = []
cnt = {0: 0, 1: 0}
for pos in order:
    t = int(targets[pos])
    if cnt[t] < 184:
        hold_pos.append(int(pos))
        cnt[t] += 1
    if cnt[0] == 184 and cnt[1] == 184:
        break
assert cnt[0] == 184 and cnt[1] == 184, f"holdout not 184/184: {cnt}"
hold_pos = np.array(sorted(hold_pos))

isic_dir = os.path.join(core.PATHS["root"], "isic")
core._ensure(isic_dir)
pd.DataFrame({"image_name": names[hold_pos], "target": targets[hold_pos]}).to_csv(
    os.path.join(isic_dir, "holdout_ids.csv"), index=False)

train_mask = np.ones(n_dedup, dtype=bool)
train_mask[hold_pos] = False
train_positions = np.where(train_mask)[0]
tp_mal = int((targets[train_positions] == 1).sum())
tp_ben = int((targets[train_positions] == 0).sum())
print(f"training pool: malignant {tp_mal} / benign {tp_ben}")
assert (tp_mal, tp_ben) == (397, 31936), "training pool counts unexpected"

deduped pool: 32701 (malignant 581 / benign 32120)
malignant margin after holdout + fixed minority: 13
training pool: malignant 397 / benign 31936


# Decode and resize once, cache 96x96 uint8 in RAM

In [7]:
t0 = time.time()
cache = np.empty((n_dedup, 96, 96, 3), dtype=np.uint8)
for i, name in enumerate(names):
    path = os.path.join(img_dir, name + ".jpg")
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.draft("RGB", (96, 96))
        im = im.resize((96, 96), Image.BILINEAR)
        cache[i] = np.asarray(im, dtype=np.uint8)
    if (i + 1) % 2000 == 0 or (i + 1) == n_dedup:
        el = time.time() - t0
        rate = (i + 1) / el
        eta = (n_dedup - (i + 1)) / rate if rate > 0 else 0
        print(f"  decoded {i + 1}/{n_dedup} ({el:.0f}s elapsed, "
              f"{rate:.0f} img/s, eta {eta:.0f}s)", flush=True)
labels = targets.copy()
print(f"decoded + resized {n_dedup} images in {time.time() - t0:.0f}s", flush=True)
print(f"cache shape {cache.shape}, {cache.nbytes / 1024**3:.2f} GiB", flush=True)

  decoded 2000/32701 (486s elapsed, 4 img/s, eta 7455s)
  decoded 4000/32701 (916s elapsed, 4 img/s, eta 6572s)
  decoded 6000/32701 (1349s elapsed, 4 img/s, eta 6004s)
  decoded 8000/32701 (1779s elapsed, 4 img/s, eta 5493s)
  decoded 10000/32701 (2204s elapsed, 5 img/s, eta 5004s)
  decoded 12000/32701 (2611s elapsed, 5 img/s, eta 4505s)
  decoded 14000/32701 (3036s elapsed, 5 img/s, eta 4055s)
  decoded 16000/32701 (3451s elapsed, 5 img/s, eta 3602s)
  decoded 18000/32701 (3861s elapsed, 5 img/s, eta 3154s)
  decoded 20000/32701 (4266s elapsed, 5 img/s, eta 2709s)
  decoded 22000/32701 (4670s elapsed, 5 img/s, eta 2272s)
  decoded 24000/32701 (5121s elapsed, 5 img/s, eta 1857s)
  decoded 26000/32701 (5585s elapsed, 5 img/s, eta 1439s)
  decoded 28000/32701 (6113s elapsed, 5 img/s, eta 1026s)
  decoded 30000/32701 (6576s elapsed, 5 img/s, eta 592s)
  decoded 32000/32701 (7014s elapsed, 5 img/s, eta 154s)
  decoded 32701/32701 (7165s elapsed, 5 img/s, eta 0s)
decoded + resized 32701 i

# ISIC normalisation stats (training pool only, no reference gate)

In [8]:
train_rows_sorted = np.sort(train_positions)
isic_mean, isic_std = core.compute_norm_stats(cache[train_rows_sorted])

assert all(np.isfinite(isic_mean)) and all(np.isfinite(isic_std)), "non-finite ISIC stats"
assert all(0.0 < m < 1.0 for m in isic_mean), "ISIC mean outside (0,1)"
assert all(0.0 < s < 1.0 for s in isic_std), "ISIC std outside (0,1)"

print("ISIC mean:", [f"{m:.6f}" for m in isic_mean])
print("ISIC std :", [f"{s:.6f}" for s in isic_std])

with open(os.path.join(isic_dir, "isic_norm.md"), "w") as f:
    f.write("# ISIC normalisation stats (training pool, 96x96 bilinear, [0,1])\n\n")
    f.write(f"- mean: {isic_mean[0]:.6f}, {isic_mean[1]:.6f}, {isic_mean[2]:.6f}\n")
    f.write(f"- std: {isic_std[0]:.6f}, {isic_std[1]:.6f}, {isic_std[2]:.6f}\n")
print("wrote", os.path.join(isic_dir, "isic_norm.md"))

ISIC mean: ['0.805821', '0.620952', '0.590992']
ISIC std : ['0.147848', '0.174740', '0.200526']
wrote /kaggle/working/isic/isic_norm.md


# ISIC datasets (indexing the cache) and nested subset sampling

In [9]:
_isic_m = torch.tensor(isic_mean, dtype=torch.float32).view(3, 1, 1)
_isic_s = torch.tensor(isic_std, dtype=torch.float32).view(3, 1, 1)


class ISICTrain(Dataset):
    def __init__(self, cache, labels, rows, loader_seed):
        self.cache = cache
        self.labels = labels
        self.rows = np.asarray(rows)
        self.aug_rng = np.random.default_rng(loader_seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        g = int(self.rows[i])
        img = self.cache[g]
        if self.aug_rng.random() < 0.5:
            img = img[:, ::-1, :]
        if self.aug_rng.random() < 0.5:
            img = img[::-1, :, :]
        k = int(self.aug_rng.integers(0, 4))
        if k:
            img = np.rot90(img, k)
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div_(255.0).permute(2, 0, 1)
        x = (x - _isic_m) / _isic_s
        return x, int(self.labels[g])


class ISICEval(Dataset):
    def __init__(self, cache, labels, rows):
        self.cache = cache
        self.labels = labels
        self.rows = np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        g = int(self.rows[i])
        img = self.cache[g]
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div_(255.0).permute(2, 0, 1)
        x = (x - _isic_m) / _isic_s
        return x, int(self.labels[g])


mal_rows = train_positions[labels[train_positions] == 1]
ben_rows = train_positions[labels[train_positions] == 0]
assert len(mal_rows) == 397 and len(ben_rows) == 31936

subset_dir = os.path.join(isic_dir, "subsets")
core._ensure(subset_dir)


def isic_subset(n_min, n_maj, data_seed, save=False):
    rng = np.random.default_rng(data_seed)
    mp = mal_rows[rng.permutation(len(mal_rows))]
    bp = ben_rows[rng.permutation(len(ben_rows))]
    if n_min > len(mp):
        raise ValueError(f"n_min {n_min} exceeds malignant pool {len(mp)}")
    if n_maj > len(bp):
        raise ValueError(f"n_maj {n_maj} exceeds benign pool {len(bp)}")
    rows = np.concatenate([mp[:n_min], bp[:n_maj]])
    if save:
        pd.DataFrame({"image_name": names[rows], "target": labels[rows]}).to_csv(
            os.path.join(subset_dir, f"isic_m{n_min}_M{n_maj}_d{data_seed}.csv"), index=False)
    return rows


holdout_loader = DataLoader(
    ISICEval(cache, labels, hold_pos), batch_size=512, shuffle=False, num_workers=0)

# Build the ISIC condition list

In [10]:
N_MIN_FIXED = 384
ISIC_RATIOS = [1, 5, 10, 25, 50]

conditions = []
print(f"{'idx':>3}  {'target r':>8}  {'n_min':>6}  {'n_maj':>6}  {'total N':>8}")
for i, r in enumerate(ISIC_RATIOS):
    n_min = N_MIN_FIXED
    n_maj = N_MIN_FIXED * r
    assert n_maj <= len(ben_rows), f"n_maj {n_maj} exceeds benign pool at r={r}"
    conditions.append((r, n_min, n_maj))
    print(f"{i:>3}  {r:>8}  {n_min:>6}  {n_maj:>6}  {n_min + n_maj:>8}")

print(f"\n{len(conditions)} conditions, {len(core.MODEL_NAMES)} models, "
      f"{len(core.SEEDS)} seeds -> "
      f"{len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)} runs")

idx  target r   n_min   n_maj   total N
  0         1     384     384       768
  1         5     384    1920      2304
  2        10     384    3840      4224
  3        25     384    9600      9984
  4        50     384   19200     19584

5 conditions, 3 models, 3 seeds -> 45 runs


# Run the ISIC sweep

In [11]:
isic_ledger_path = os.path.join(core.PATHS["results"], "isic_runs_ledger.csv")


def _read_isic_ledger():
    if os.path.exists(isic_ledger_path):
        return pd.read_csv(isic_ledger_path)
    return pd.DataFrame(columns=core.LEDGER_COLUMNS)


def _append_isic_ledger(row):
    df = _read_isic_ledger()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    core._ensure(core.PATHS["results"])
    df.to_csv(isic_ledger_path, index=False)


def _isic_run_id(model, n_min, n_maj, s):
    return f"isic__{model}__m{n_min}_M{n_maj}__d{s}_i{s}_l{s}"


def _save_holdout_logits(run_id, y_true, logit_diff):
    core._ensure(core.PATHS["logits"])
    path = os.path.join(core.PATHS["logits"], f"{run_id}_holdout.csv")
    pd.DataFrame({"y_true": np.asarray(y_true).astype(int),
                  "logit_diff": np.asarray(logit_diff).astype(np.float32)}).to_csv(path, index=False)


done = 0
for model in core.MODEL_NAMES:
    for (r, n_min, n_maj) in conditions:
        for s in core.SEEDS:
            run_id = _isic_run_id(model, n_min, n_maj, s)
            done += 1
            tag = f"[isic][{model}][m={n_min} M={n_maj} r={r}][seed={s}]"

            ledg = _read_isic_ledger()
            hit = ledg[(ledg["run_id"] == run_id)
                       & (ledg["status"] == "success")
                       & (ledg["config_hash"] == core.CONFIG_HASH)]
            if len(hit) > 0:
                print(f"[{done:>2}/45] {tag} reuse, no retrain -> {run_id}")
                continue

            n_total = n_min + n_maj
            actual_ratio = n_maj / n_min
            exposures = core.STEP_BUDGET * core.BATCH_SIZE / n_total
            try:
                rows = isic_subset(n_min, n_maj, s, save=True)
                train_loader = DataLoader(
                    ISICTrain(cache, labels, rows, s),
                    batch_size=core.BATCH_SIZE, shuffle=True, drop_last=True,
                    num_workers=0, generator=torch.Generator().manual_seed(int(s)))

                core.set_seed(s)
                net = core.build_model(model)

                def _log_fn(step, loss):
                    core.log(f"{tag} step {step}/{core.STEP_BUDGET} loss={loss:.4f}", stage="isic")

                net, curve, actual_steps, eff_epochs = core.train_model(
                    net, train_loader, core.HP[model], core.STEP_BUDGET, _log_fn)

                y_hold, d_hold = core.dump_logits(net, holdout_loader)
                _save_holdout_logits(run_id, y_hold, d_hold)
                core.save_curve(run_id, curve)

                _append_isic_ledger({
                    "run_id": run_id, "condition_id": f"isic_m{n_min}_M{n_maj}",
                    "protocol": "isic", "model": model, "n_min": n_min, "n_maj": n_maj,
                    "target_ratio": r, "actual_ratio": actual_ratio,
                    "data_seed": s, "init_seed": s, "loader_seed": s,
                    "step_budget": core.STEP_BUDGET, "actual_steps": actual_steps,
                    "effective_epochs": eff_epochs, "exposures_per_image": exposures,
                    "status": "success", "config_hash": core.CONFIG_HASH, "timestamp": core._now(),
                })
                print(f"[{done:>2}/45] {tag} eff_epochs={eff_epochs:.1f} -> {run_id}")

            except Exception as exc:
                import traceback
                core.log(f"{tag} FAILED: {exc}", stage="isic")
                core.log(traceback.format_exc(), stage="isic", also_print=False)
                _append_isic_ledger({
                    "run_id": run_id, "condition_id": f"isic_m{n_min}_M{n_maj}",
                    "protocol": "isic", "model": model, "n_min": n_min, "n_maj": n_maj,
                    "target_ratio": r, "actual_ratio": actual_ratio,
                    "data_seed": s, "init_seed": s, "loader_seed": s,
                    "step_budget": core.STEP_BUDGET, "actual_steps": 0,
                    "effective_epochs": 0.0, "exposures_per_image": exposures,
                    "status": "failed", "config_hash": core.CONFIG_HASH, "timestamp": core._now(),
                })
                print(f"[{done:>2}/45] {tag} FAILED")

print("sweep calls completed:", done)

[14:52:50] logistic parameter count: 55298
[14:53:09] [isic][logistic][m=384 M=384 r=1][seed=21] step 250/5000 loss=0.4778
[14:53:26] [isic][logistic][m=384 M=384 r=1][seed=21] step 500/5000 loss=0.7045
[14:53:43] [isic][logistic][m=384 M=384 r=1][seed=21] step 750/5000 loss=0.4996
[14:54:01] [isic][logistic][m=384 M=384 r=1][seed=21] step 1000/5000 loss=1.0563
[14:54:18] [isic][logistic][m=384 M=384 r=1][seed=21] step 1250/5000 loss=0.5437
[14:54:35] [isic][logistic][m=384 M=384 r=1][seed=21] step 1500/5000 loss=0.4661
[14:54:52] [isic][logistic][m=384 M=384 r=1][seed=21] step 1750/5000 loss=0.4516
[14:55:09] [isic][logistic][m=384 M=384 r=1][seed=21] step 2000/5000 loss=0.5387
[14:55:27] [isic][logistic][m=384 M=384 r=1][seed=21] step 2250/5000 loss=0.3173
[14:55:44] [isic][logistic][m=384 M=384 r=1][seed=21] step 2500/5000 loss=0.3345
[14:56:01] [isic][logistic][m=384 M=384 r=1][seed=21] step 2750/5000 loss=0.3837
[14:56:18] [isic][logistic][m=384 M=384 r=1][seed=21] step 3000/5000 

# Session summary from the ISIC ledger 

In [12]:
if not os.path.exists(isic_ledger_path):
    raise FileNotFoundError("no ISIC ledger yet; run the sweep cell first")

rows = _read_isic_ledger()
order = {"failed": 0, "success": 1}
rows = rows.assign(_rank=rows["status"].map(order).fillna(-1))
dedup = rows.sort_values("_rank").drop_duplicates("run_id", keep="last").drop(columns="_rank")
n_success = int((dedup["status"] == "success").sum())
n_failed = int((dedup["status"] == "failed").sum())
expected = len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)

print(f"ISIC: {n_success} success, {n_failed} failed, {expected - n_success} pending of {expected}")

failed_rows = dedup[dedup["status"] == "failed"]
if len(failed_rows):
    print("\nfailed runs:")
    for _, rr in failed_rows.iterrows():
        print(f"  {rr['run_id']}  (model={rr['model']} "
              f"n_min={rr['n_min']} n_maj={rr['n_maj']} seed={rr['data_seed']})")

core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "05_isic.md")
lines = [
    "# Notebook 05: ISIC external check session summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    "## Data construction",
    "- deduped pool: 32,701 (malignant 581 / benign 32,120), dropping every image_name_2",
    "- holdout: 184 / 184 = 368 images, seed 20200, image-disjoint after dedup",
    "- training pool: malignant 397 / benign 31,936",
    "- fixed training minority: 384 malignant",
    f"- ratios: {ISIC_RATIOS}, n_maj = 384 * r = {[384 * r for r in ISIC_RATIOS]}",
    "- resize: 96x96 bilinear, decoded and cached once",
    f"- ISIC norm mean: {isic_mean[0]:.6f}, {isic_mean[1]:.6f}, {isic_mean[2]:.6f}",
    f"- ISIC norm std: {isic_std[0]:.6f}, {isic_std[1]:.6f}, {isic_std[2]:.6f}",
    "",
    "## Runs",
    f"- expected unique runs: {expected} (5 conditions x 3 models x 3 seeds)",
    f"- success: {n_success}",
    f"- failed: {n_failed}",
    f"- pending: {expected - n_success}",
    "",
    "## Caveat",
    "- the holdout is image-disjoint after duplicate removal; patient-level separation is "
    "not enforced by design. ISIC is a secondary consistency check, not a leakage-controlled benchmark.",
    "",
    "## Failed runs",
]
if len(failed_rows):
    for _, rr in failed_rows.iterrows():
        lines.append(f"- {rr['run_id']} (model={rr['model']} "
                     f"n_min={rr['n_min']} n_maj={rr['n_maj']} seed={rr['data_seed']})")
else:
    lines.append("- none")
lines.append("")
with open(summary_path, "w") as f:
    f.write("\n".join(lines))
print("\nwrote", summary_path)

ISIC: 45 success, 0 failed, 0 pending of 45

wrote /kaggle/working/summaries/05_isic.md


# Persistence 

In [13]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, the ledger and logits will not survive session end.")

Starting upload for file __notebook__.ipynb


100%|██████████| 207k/207k [00:00<00:00, 321kB/s]


Upload successful: __notebook__.ipynb (207KB)
Starting upload for file frozen_config.md


100%|██████████| 431/431 [00:00<00:00, 533B/s]


Upload successful: frozen_config.md (431B)
Starting upload for file summaries.zip


...

Upload successful: results.zip (5KB)
Starting upload for file training_curves.zip


100%|██████████| 109k/109k [00:00<00:00, 117kB/s]


Upload successful: training_curves.zip (109KB)
Starting upload for file logits.zip


100%|██████████| 69.0M/69.0M [00:02<00:00, 26.2MB/s]


Upload successful: logits.zip (69MB)
pushed results store to hrushisanap/datawise-results
